`RecallDataset` is a dictionary of arrays describing study lists and recall sequences, with one row per trial shared across all fields. Study columns describe presentations and recall columns describe responses; their widths can differ.

## Dataset structure

In [1]:
#| code-summary: "1 — Import packages"

from pathlib import Path
from tempfile import TemporaryDirectory

import h5py
import jax.numpy as jnp
import numpy as np
from nbdev.showdoc import show_doc

from jaxcmr.helpers import load_data, make_dataset, save_dict_to_hdf5
from jaxcmr.typing import RecallDataset


In [2]:
#| code-summary: "2 — Show the current dataset definition"

show_doc(RecallDataset)


---

### RecallDataset



*Trial-indexed arrays for free or serial recall experiments.

The required fields are ``subject``, ``listLength``, ``pres_itemnos``,
and ``recalls``. Their annotated shapes and integer dtypes appear below;
zeros pad item and recall sequences.

``pres_itemids`` identifies items across lists and is required by semantic
models and analyses, but not by every recall dataset. Other named fields
are optional. Additional fields may contain arrays of any dtype or shape,
such as valence or LPP measurements. A consumer that needs an optional or
additional field must check that the dataset contains it. Every field must
have a leading trial dimension; non-trial metadata belongs outside this dict.*

| Field | Required | Shape | Meaning |
|---|---|---|---|
| `subject` | Yes | `(trials, 1)` | Participant identifier. |
| `listLength` | Yes | `(trials, 1)` | Valid study events, including repeated presentations. |
| `pres_itemnos` | Yes | `(trials, study slots)` | Within-list item codes; repetitions retain the same code. |
| `recalls` | Yes | `(trials, response slots)` | Study positions of recalled items, in response order. |
| `pres_itemids` | No | `(trials, study slots)` | Item identities in the shared stimulus pool. |
| `rec_itemids` | No | `(trials, response slots)` | Shared-pool identities of recalled items. |

## Load and inspect data

This example uses the [war-photo-recall dataset](../templates.qmd#download-the-shared-inputs). Set `data_path` to its location in your working folder.

In [3]:
#| code-summary: "3 — Load the dataset and inspect its fields"

data_path = "data/Dupertuys2026.h5"
data = load_data(data_path)

for name, values in data.items():
    print(f"{name}: shape={values.shape}, dtype={values.dtype}")


listLength: shape=(492, 1), dtype=int32
list_number: shape=(492, 1), dtype=int32
list_type: shape=(492, 1), dtype=int32
mean_arousal: shape=(492, 12), dtype=float32
mean_valence: shape=(492, 12), dtype=float32
pres_itemids: shape=(492, 12), dtype=int32
pres_itemnos: shape=(492, 12), dtype=int32
rec_itemids: shape=(492, 12), dtype=int32
recalls: shape=(492, 12), dtype=int32
subject: shape=(492, 1), dtype=int32
valence: shape=(492, 12), dtype=int32


In [4]:
#| code-summary: "4 — Inspect one trial and count its recorded recalls"

trial_idx = 0
print("Participant:", int(data["subject"][trial_idx, 0]))
print("List length:", int(data["listLength"][trial_idx, 0]))
print("Presented item codes:", data["pres_itemnos"][trial_idx])
print("Recalls:", data["recalls"][trial_idx])
print("Non-padding recalls:", int(np.count_nonzero(data["recalls"][trial_idx] > 0)))


Participant: 1
List length: 12
Presented item codes: [ 1  2  3  4  5  6  7  8  9 10 11 12]
Recalls: [ 2  8  5  6 12  9  0  0  0  0  0  0]
Non-padding recalls: 6


## Understand the codes

A positive recall code `p` identifies study position `p`; use `p - 1` when indexing a Python array. Zero pads unused study and response slots.

In [5]:
#| code-summary: "5 — Translate recall positions into within-list and global item IDs"

positions = data["recalls"][trial_idx]
positions = positions[positions > 0]
print("Recall positions:", positions)
print("Within-list item codes:", data["pres_itemnos"][trial_idx, positions - 1])
print("Global item IDs:", data["pres_itemids"][trial_idx, positions - 1])


Recall positions: [ 2  8  5  6 12  9]
Within-list item codes: [ 2  8  5  6 12  9]
Global item IDs: [66 23 25 59 41 22]


For study codes `[1, 2, 1]`, positions 1 and 3 identify the same item: recall position 3 maps to item code 1. Dataset simulation reports its first-occurrence position, 1.

## Additional fields

| Alignment | Examples | Keep aligned with |
|---|---|---|
| Trial | `session`, `list_type` | Trial rows. |
| Study event | `valence`, `pres_itemids` | Trial rows and study slots. |
| Recall event | `irt`, `rec_itemids` | Trial rows and response slots. |

Vocabulary-wide features and distance matrices remain separate inputs.

## Construct, save and reload

`make_dataset()` constructs the four core fields. The example below uses the selected trial's recorded participant and list length.

In [6]:
#| code-summary: "6 — Construct one trial and save and reload its arrays"

one_trial = make_dataset(
    recalls=data["recalls"][trial_idx],
    pres_itemnos=data["pres_itemnos"][trial_idx],
    listLength=int(data["listLength"][trial_idx, 0]),
    subject=int(data["subject"][trial_idx, 0]),
)
with TemporaryDirectory() as folder:
    path = str(Path(folder) / "example.h5")
    save_dict_to_hdf5(one_trial, path)
    with h5py.File(path, "r") as stored:
        print("Stored recalls:", stored["data/recalls"].shape)
    reloaded = load_data(path)
    print("Loaded recalls:", reloaded["recalls"].shape)
    print("Arrays preserved:", all(
        bool(jnp.array_equal(one_trial[name], reloaded[name])) for name in one_trial
    ))


Stored recalls: (12, 1)
Loaded recalls: (1, 12)
Arrays preserved: True


`save_dict_to_hdf5()` writes transposed arrays under `/data`; `load_data()` transposes them back into trial rows and returns JAX arrays. This is the helpers' storage convention, not a requirement of HDF5.

Continue with [Recall Analyses](analyses.ipynb).